# Lean-43 — The covering bound C(24,14,4) ≥ 20, certified in Lean

**Navigation** : [Index](README.md) | [<< Lean-42 Certified arithmetic independence](Lean-42-Independance-Arithmetique-BusyBeaver-Lean.ipynb) | [English version](Lean-43-Borne-Couverture-C24-14-4-Lean_en.ipynb)

**Lake** : [`covering_lean/`](covering_lean/README.md) — pinned consumer of the `covering` package from [jamesyc/covering](https://github.com/jamesyc/covering).

**Source** : paper *A structural lower bound for C(24,14,4)* (Apache-2.0), archived in the shared shelf `Bibliographie IA/Automata/Complexity Theory and Hardness/2026 - Chang - A structural lower bound for C(24,14,4) (github jamesyc-covering, Apache-2.0, Lean formalise).pdf`. Series issue: #27477, milestone 3.

**Objectives**

- read a Lean **consumer lake**: where the upstream package lives, at which commit, with which toolchain;
- tell a **proven lower bound** apart from a **determined covering number** — the most expensive confusion in this chapter;
- read an **axiom audit** (`#print axioms`) and confront it with the list the upstream grants itself;
- connect a combinatorial bound to the **Schönheim recurrence**, and see what it propagates and what it does not prove.

**Prerequisites** : basic combinatorics (families of blocks, covering designs) and Lean 4 (modules, `lake`, `#print axioms`). The notebook [Lean-42](Lean-42-Independance-Arithmetique-BusyBeaver-Lean.ipynb) introduced the consumer-lake pattern; this one continues with a combinatorial example.

**Estimated duration** : 45 minutes of reading, plus the time of a Lean build (a few minutes on an already-built lake).

## Plan

1. **The consumer lake** : where the upstream package lives, at which commit, with which toolchain.
2. **The pin** : re-read the manifest and check that the revision holds.
3. **The build** : build the witness, read what `lake` actually did.
4. **The axiom audit** : `#print axioms` on the two theorems, and the mechanical verdict.
5. **The numbers of the paper** : bounds, the upstream table, propagation by Schönheim.
6. **Reading** : what « C(24,14,4) ≥ 20 » establishes, and what it does not say.
7. **Exercises** : three, to be written in the stub cells.

## 1. The consumer lake

The `covering_lean` lake defines **nothing**: it pins the `covering` package of the upstream repository and imports its theorems. The clause of the `lakefile.lean` is direct:

```lean
require «covering» from git
  "https://github.com/jamesyc/covering.git" @ "e28962cff844f155ca5a5cfb22563083369b490e"
```

No fork, no copy, no adaptation: Lake clones the upstream repository at the announced commit and loads the package exactly as published.

The upstream declares **three** libraries, and they do not play the same role:

| Library | Role | Content |
|---|---|---|
| `Covering` | the proof | the modules that build the argument, up to `Covering/Main.lean` |
| `Challenge` | the audit artefact | the two statements **with `sorry`** — the question asked, not the answer |
| `Solution` | the answer | the same statements, proved, for the submission to the Palomar registry |

The consumer imports `Covering` only. It touches neither `Challenge` (the `sorry` there are deliberate: that is the specification) nor `Solution` (redundant with `Covering.Main`).

The toolchain of the lake is **v4.35.0-rc3**, the one of the upstream package, which pins Mathlib to the same tag. This is not an oversight: the series now counts three lakes with three toolchains (`formal_logic_lean` on 4.33.1, `busybeaver_lean` on v4.35.0, `covering_lean` on v4.35.0-rc3). One lake per toolchain is the shape Lake provides for; the three live side by side, with no dependency between them.

The next cell locates the lake, computes the WSL path, and compares the SHA-256 digests of the build files between the repository and the lake actually used: a build that succeeds in that lake is worth something for the repository **only** if the four digests coincide.

In [1]:
import hashlib
import os
import pathlib
import subprocess

NB_DIR = pathlib.Path.cwd()
REPO_LAKE = NB_DIR / "covering_lean"
PIN_ATTENDU = "e28962cff844f155ca5a5cfb22563083369b490e"


def wslpath(p):
    # Chemin Windows -> chemin WSL, calcule par WSL lui-meme.
    r = subprocess.run(["wsl", "-e", "wslpath", "-a", str(p).replace(chr(92), "/")],
                       capture_output=True, text=True, encoding="utf-8", errors="replace")
    return r.stdout.strip()


def wsl_sh(commande, tentatives=3):
    # Execute une commande shell dans WSL et rend sa sortie standard.
    # WSL peut refuser une invocation transitoirement (machine virtuelle occupee
    # par un build borne) : on rejoue, et on leve si l'echec est reel plutot
    # que de rendre une sortie vide que l'on croirait lire.
    dernier = None
    for _ in range(tentatives):
        r = subprocess.run(["wsl", "-e", "bash", "-lc", commande],
                           capture_output=True, text=True, encoding="utf-8", errors="replace")
        if r.returncode == 0:
            return r.stdout
        dernier = r
    raise RuntimeError(f"wsl a echoue (rc={dernier.returncode}) : {commande}\n"
                       f"{dernier.stderr.strip()}")


LAKE_OVERRIDE = os.environ.get("COVERING_LAKE_WSL")
LAKE_DIR = LAKE_OVERRIDE or wslpath(REPO_LAKE)
LEAN_EXEC = wslpath(NB_DIR.parents[2] / "scripts" / "lean" / "lean_exec.py")


def lean_exec(commande, timeout=900):
    # Lance `commande` dans le lake via l'organe borne ; rend (code, sortie, journal).
    r = subprocess.run(
        ["wsl", "-e", "bash", "-lc",
         f"cd {LAKE_DIR} && python3 {LEAN_EXEC} run --timeout {timeout} -- {commande}"],
        capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=timeout + 120)
    journal = [l for l in r.stderr.splitlines() if l.startswith("[lean_exec]")]
    return r.returncode, r.stdout, journal


FICHIERS_BUILD = ["lean-toolchain", "lakefile.lean", "lake-manifest.json", "CoveringConsumer.lean"]

# Tout est accumule puis imprime en UNE fois : un seul flush stdout, donc une
# seule sortie commitee -- le jumeau EN ne peut pas decouper autrement.
lignes = ["lake utilise : " + ("copie (COVERING_LAKE_WSL)" if LAKE_OVERRIDE
                               else "covering_lean du depot")]
cote_wsl = wsl_sh(f"cd {LAKE_DIR} && sha256sum " + " ".join(FICHIERS_BUILD)).splitlines()
empreintes_wsl = {l.split()[1]: l.split()[0] for l in cote_wsl if l.strip()}
identiques = 0
for nom in FICHIERS_BUILD:
    h_depot = hashlib.sha256((REPO_LAKE / nom).read_bytes()).hexdigest()
    h_lake = empreintes_wsl.get(nom, "absent")
    identiques += h_depot == h_lake
    lignes.append(f"  {nom:22s} depot {h_depot[:12]}  lake {h_lake[:12]}  "
                  f"{'identique' if h_depot == h_lake else 'DIFFERENT'}")
lignes.append(f"{identiques}/{len(FICHIERS_BUILD)} fichiers identiques"
              + ("  (comparaison triviale : meme dossier)" if not LAKE_OVERRIDE else ""))

# Le manifeste peut annoncer un pin ; la tete du clone reellement construit le
# prouve. Les deux sont lus : le manifeste a la cellule suivante, la tete ici.
tete = wsl_sh(f"git -C {LAKE_DIR}/.lake/packages/covering rev-parse HEAD").strip()
lignes.append("tete du clone amont : " + tete)
lignes.append("pin tenu (clone)    : " + str(tete == PIN_ATTENDU))
lignes.append("toolchain           : " + (REPO_LAKE / "lean-toolchain").read_text().strip())
print("\n".join(lignes))

lake utilise : copie (COVERING_LAKE_WSL)
  lean-toolchain         depot bc84812c9448  lake bc84812c9448  identique
  lakefile.lean          depot c67a35e583ad  lake c67a35e583ad  identique
  lake-manifest.json     depot 1871390ce742  lake 1871390ce742  identique
  CoveringConsumer.lean  depot 735b5d878c01  lake 735b5d878c01  identique
4/4 fichiers identiques
tete du clone amont : e28962cff844f155ca5a5cfb22563083369b490e
pin tenu (clone)    : True
toolchain           : leanprover/lean4:v4.35.0-rc3


### Reading the set-up

Four lines, one per build file: each shows the digest on the repository side and the digest on the lake side. All of them must read `identique` — otherwise the lake that was built is a different object than the one in the repository (a forgotten copy, an intermediate state). When the lake is a copy prepared in advance (the `COVERING_LAKE_WSL` variable), the comparison is real; on the repository lake it is trivially true, and the cell says so rather than letting it be believed.

Two lines close the blind spot of the manifest: `git rev-parse HEAD` on the upstream clone gives the revision **actually checked out**, and `pin tenu (clone)` must read `True`. A manifest may announce a pin the clone does not have; the head of the clone cannot. The toolchain shown is the one `lake` will use — `v4.35.0-rc3`.

## 2. The pin, re-read in the manifest

`lake update` resolved the `from git` clause into a revision written in `lake-manifest.json`. That file is the contract: every later build starts from those revisions, not from the upstream branch.

The cell reads the manifest and looks for the `covering` package. We expect the commit `e28962cf…` — the one on which the upstream publishes its package, its audit and its paper. A manifest carrying any other revision would be a silent drift: the build would succeed, but on different content.

In [2]:
# Le manifeste Lake fige le commit exact du paquet : c'est la preuve du pin.
import json

manifeste = json.loads((REPO_LAKE / "lake-manifest.json").read_text(encoding="utf-8"))
# `covering` est la dependance directe ; les autres paquets viennent de ses transitives.
paquet = next(p for p in manifeste["packages"] if p["name"] == "covering")
print("paquet      :", paquet["name"])
print("rev         :", paquet["rev"])
print("url         :", paquet["url"])
print("pin tenu    :", paquet["rev"] == PIN_ATTENDU)
print("sous-dir lu :", paquet.get("subDir"))
print("dependances :", ", ".join(sorted(p["name"] for p in manifeste["packages"]
                                       if p["name"] != "covering")))

paquet      : covering
rev         : e28962cff844f155ca5a5cfb22563083369b490e
url         : https://github.com/jamesyc/covering.git
pin tenu    : True
sous-dir lu : None
dependances : Cli, LeanSearchClient, Qq, aesop, batteries, importGraph, mathlib, plausible, proofwidgets


### Reading the pin

Four values to confront: `rev` must equal the expected pin (`pin tenu : True`), `url` must designate `jamesyc/covering`, `sous-dir lu` must read `None` — the package sits at the root of the upstream repository, unlike the `PA196` package consumed by the Lean-42 sibling, which lives in a subdirectory — and the dependency list must be the one the upstream package declares. Those dependencies — Mathlib, then `aesop`, `batteries`, `Qq`, `Cli`, `plausible`, `proofwidgets`, `import-graph`, `LeanSearchClient` — are resolved by the upstream package itself: this lake does not re-declare them, so it cannot make them diverge.

Mathlib is here at `v4.35.0-rc3`, the revision the upstream pins: the consistency between the package and its base library holds without any intervention of ours.

## 3. The build

The cell runs `lake build` in the lake, through the organ `scripts/lean/lean_exec.py`: machine-wide cap, confinement of the process tree, measured budget. It is the same organ as the other Lean notebooks of the series; no `lake` is launched by hand.

On a fresh lake, two commands come first: `lake update` (resolution of the pin and writing of the manifest) then `lake exe cache get` (Mathlib oleans of the upstream revision). Without the cache, Mathlib would be compiled from the sources — several hours for content the upstream already publishes. The notebook takes them for granted: its object is the build of the witness, not the preparation of the lake.

A first build then pays for the compilation of the upstream package — a little more than two thousand modules — and the following ones are incremental. The number of tasks `lake` prints is therefore a state of the cache, not a property of the result: that is why the cell keeps only the conclusion line.

In [3]:
# Build du temoin : incremental une fois les oleans en place.
code_retour, sortie, journal = lean_exec("lake build", timeout=5400)
# Seule la ligne de conclusion est retenue : le detail des taches (et leur
# duree) depend de l'etat du cache, pas du resultat.
utiles = [l for l in sortie.splitlines() if l.startswith("Build completed successfully")]
print("\n".join(utiles[-1:]))
print("build reussi :", code_retour == 0)
# Le temps de build varie d'un run a l'autre : normalise, sinon le jumeau EN
# ne peut pas porter la meme sortie (garde twin-parity).
import re
print("\n".join(re.sub(r"duration=[0-9.]+s", "duration=mesuree", l) for l in journal))
print("code de retour :", code_retour)

Build completed successfully (2180 jobs).
build reussi : True
[lean_exec] ok exit=0 duration=mesuree orphans=0
[lean_exec] child_exit_code: 0
code de retour : 0


### Reading the build

The **return code** must be zero: a `lake build` that fails returns a non-zero code and error lines. The **organ's journal** carries the admission measure and the « zero orphan descendant » postcondition; its duration is normalised, because it depends on the machine rather than on the result.

The conclusion line — `Build completed successfully` — is the one that counts: it means that the whole set of targets was built, the upstream package as well as the consumer module. An incremental build may print no task at all: that is the sign that the `.olean` files are up to date, not that nothing was checked.

The upstream package publishes its own checks: the `Solution` module carries the `#print axioms` of its two theorems, and the `comparator.json` file the list of axioms the upstream grants itself. The next section confronts our audit with that list, rather than with a list invented on the spot.

## 4. The axiom audit

Lean can be asked, for each theorem, for the list of axioms its proof depends on. That is the measurement that tells a complete proof apart from a proof that leans on `sorry` (`sorryAx`) or on native evaluation (`Lean.ofReduceBool`, through `native_decide`).

The `CoveringConsumer.lean` module of the lake writes two `#print axioms`: `Covering.covering_24_14_4` and `Covering.covering_25_15_5`. The cell re-elaborates the module with `lake env lean` — the imports are resolved by the lake — and reads the axiom lines.

The module also carries two constant-body witnesses (`def witness_covering_24_14_4`, `def witness_covering_25_15_5`): the inferred type is the upstream statement, without copying it. They are not proof objects — they are the declarations that the CI axiom organ enumerates for this lake; without them the gate would inspect nothing.

The verdict is then mechanical, not declarative: the next cell compares the quoted set with the whitelist `propext`, `Classical.choice`, `Quot.sound`, with the forbidden list `sorryAx`, `Lean.ofReduceBool`, `Lean.trustCompiler`, **and** with the list the upstream publishes itself in `comparator.json`.

In [4]:
# Audit d'axiomes : `lake env lean` re-elabore le module et imprime chaque
# ligne `#print axioms` ecrite dans CoveringConsumer.lean.
code_retour, sortie, journal = lean_exec("lake env lean CoveringConsumer.lean", timeout=1800)
lignes_axiomes = [l.strip() for l in sortie.splitlines() if "axioms" in l.lower()]
print("\n".join(lignes_axiomes))
print("code de retour :", code_retour)

'Covering.covering_24_14_4' depends on axioms: [propext, Classical.choice, Quot.sound]
'Covering.covering_25_15_5' depends on axioms: [propext, Classical.choice, Quot.sound]
code de retour : 0


### Reading the axiom lines

Two lines, one per theorem, and the same list in both cases: `propext`, `Classical.choice`, `Quot.sound`. These are the three axioms of the Lean kernel and of Mathlib — propositional extensionality, classical choice, quotients. None of them is a hole in a proof: `sorryAx` would be one, and it is not in the list.

The next cell does not settle for this reading: it confronts the measured set with the list the upstream publishes in `comparator.json`, that is, with what the upstream itself declares acceptable for its submission. That is a cross-check, not a second reading of the same list.

In [5]:
AUTORISES = {"propext", "Classical.choice", "Quot.sound"}
INTERDITS = {"sorryAx", "Lean.ofReduceBool", "Lean.trustCompiler"}


def axiomes_cites(lignes):
    # Extrait l'ensemble des axiomes cites par les lignes `#print axioms`.
    trouves = set()
    for ligne in lignes:
        if "depends on axioms:" in ligne:
            liste = ligne.split("depends on axioms:")[-1].strip().strip("[]")
            trouves |= {a.strip() for a in liste.split(",") if a.strip()}
        # « does not depend on any axioms » : ensemble vide, rien a ajouter.
    return trouves


cites = axiomes_cites(lignes_axiomes)
print("axiomes cites :", sorted(cites) if cites else "(aucun)")
print("hors liste blanche :", sorted(cites - AUTORISES) or "(aucun)")
print("interdits presents :", sorted(cites & INTERDITS) or "(aucun)")
print("audit concluant :", cites <= AUTORISES and not (cites & INTERDITS))

# Controle croise : la liste que l'amont publie lui-meme pour sa soumission.
comparateur_texte = wsl_sh(f"cat {LAKE_DIR}/.lake/packages/covering/comparator.json")
if not comparateur_texte.strip():
    raise RuntimeError("comparator.json vide : la lecture amont a echoue")
comparateur = json.loads(comparateur_texte)
autorises_amont = set(comparateur["permitted_axioms"])
print("liste de l'amont :", sorted(autorises_amont))
print("concordance avec l'amont :", cites <= autorises_amont)
print("theoremes declares :", " | ".join(comparateur["theorem_names"]))

axiomes cites : ['Classical.choice', 'Quot.sound', 'propext']
hors liste blanche : (aucun)
interdits presents : (aucun)
audit concluant : True
liste de l'amont : ['Classical.choice', 'Quot.sound', 'propext']
concordance avec l'amont : True
theoremes declares : Covering.Palomar.covering_24_14_4_lower_bound | Covering.Palomar.covering_25_15_5_lower_bound


### Reading the audit

Three verdicts are read here, and they are distinct.

`audit concluant : True` is the only acceptable outcome as far as the kernel is concerned: the quoted set is a subset of the whitelist, and no forbidden axiom appears. A `sorryAx` would signal a proof left empty upstream; a `Lean.ofReduceBool` would signal a call to `native_decide`, whose trust rests on the compiler rather than on the kernel.

`concordance avec l'amont : True` is an **independent** check: the reference list is not the one written by hand above, it is the one in the `comparator.json` file published by the upstream. A gap between the two would signal that the upstream grants itself more than the Lean kernel requires — or that the file in the repository and the built package are not the same object.

`theoremes declares` names the two statements the upstream submits to the registry: `Covering.Palomar.covering_24_14_4_lower_bound` and `Covering.Palomar.covering_25_15_5_lower_bound`. They are the same statements, written in the usual `Finset` formulation, as those that the module `Covering/Main.lean` proves and that the notebook reads below.

## 5. The numbers of the paper

The result is a theorem, but its statement quotes numbers: 24 points, blocks of 14, 4-subsets to cover, and the bound 20. The cell reads these numbers **in the consumed package**, at the pinned revision — the bounds table is parsed from the upstream `README.md` and the constants are collected in `Covering/Main.lean` — then it computes the propagation through the Schönheim recurrence.

It re-proves nothing: the proof is the upstream one, consumed by the lake. Confronting a statement with its data is reading hygiene, not a claim of result.

In [6]:
# Les nombres, lus dans le paquet consomme a la revision epinglee : le tableau
# des bornes est parse depuis le README amont, pas recopie depuis le texte.
import re

texte_amont = wsl_sh(f"cat {LAKE_DIR}/.lake/packages/covering/README.md")
if not texte_amont.strip():
    raise RuntimeError("README amont vide : la lecture amont a echoue")
bornes = {}
for ligne in texte_amont.splitlines():
    m = re.match(r"\|\s*C\((\d+),\s*(\d+),\s*(\d+)\)\s*\|\s*(\d+)\s*\|\s*\*\*(\d+)\*\*\s*\|\s*(\d+)\s*\|",
                 ligne)
    if m:
        v, k, t, avant, apres, meilleur = (int(x) for x in m.groups())
        bornes[(v, k, t)] = (avant, apres, meilleur)

print("bornes lues dans le tableau amont :")
for (v, k, t), (avant, apres, meilleur) in sorted(bornes.items()):
    print(f"  C({v},{k},{t}) : borne precedente {avant} -> borne prouvee {apres}"
          f" | meilleure couverture connue {meilleur}")

# Les constantes des enonces, relevees dans le module de l'amont.
main_lean = wsl_sh(f"cat {LAKE_DIR}/.lake/packages/covering/Covering/Main.lean")
for motif, libelle in [
    (r"Finset \(Fin 24\)", "point set Fin 24"),
    (r"card = 14", "blocs de 14"),
    (r"card = 4 ", "sous-ensembles de 4"),
    ("20 ≤ ", "borne 20"),
    (r"Finset \(Fin 25\)", "point set Fin 25"),
    (r"card = 15", "blocs de 15"),
    (r"card = 5 ", "sous-ensembles de 5"),
    ("34 ≤ ", "borne 34"),
]:
    print(f"enonce contient {libelle:22s} :", bool(re.search(motif, main_lean)))

# La recurrence de Schonheim : C(v,k,t) >= ceil(v/k * C(v-1,k-1,t-1)).
def schoenheim(v, k, borne_interieure):
    # Plus petit entier superieur ou egal a (v * borne_interieure) / k.
    return -(-v * borne_interieure // k)


propages = [(26, 16, 6, 34), (27, 17, 7, 56), (28, 18, 8, 89)]
lus = re.search(r"improves C\(26,16,6\) to (\d+), C\(27,17,7\) to (\d+) "
                r"and C\(28,18,8\) to (\d+)", texte_amont)
lus = [int(x) for x in lus.groups()] if lus else []
print("propagation par Schonheim (25,15,5) -> 34 :", schoenheim(25, 15, 20))
for (v, k, t, base), annonce in zip(propages, lus + [None] * 3):
    calcule = schoenheim(v, k, base)
    print(f"  C({v},{k},{t}) : {base} -> {calcule} | amont annonce {annonce}"
          + (" | concordance : " + str(calcule == annonce) if annonce else ""))

bornes lues dans le tableau amont :
  C(24,14,4) : borne precedente 19 -> borne prouvee 20 | meilleure couverture connue 23
  C(25,15,5) : borne precedente 32 -> borne prouvee 34 | meilleure couverture connue 42
enonce contient point set Fin 24       : True
enonce contient blocs de 14            : True
enonce contient sous-ensembles de 4    : True
enonce contient borne 20               : True
enonce contient point set Fin 25       : True
enonce contient blocs de 15            : True
enonce contient sous-ensembles de 5    : True
enonce contient borne 34               : True
propagation par Schonheim (25,15,5) -> 34 : 34
  C(26,16,6) : 34 -> 56 | amont annonce 56 | concordance : True
  C(27,17,7) : 56 -> 89 | amont annonce 89 | concordance : True
  C(28,18,8) : 89 -> 139 | amont annonce 139 | concordance : True


### Reading the numbers

Three readings, three different scopes.

**The upstream table.** The bounds sit there in three columns: the lower bound previously proven, the lower bound proven by the package, and the best covering **known**. For C(24,14,4): 19 → 20, the best known construction requiring 23 blocks. For C(25,15,5): 32 → 34, best known construction requiring 42 blocks. The gain is **one** block on the first, **two** on the second — and that is exactly what the propagation by Schönheim produces, as seen below.

**The constants of the statements.** The eight `enonce contient …` lines must all read `True`: that is the re-reading of the statement in its source, not in a comment. The upstream module does say `Fin 24`, blocks of 14, 4-subsets, a bound `20 ≤`, then `Fin 25`, blocks of 15, 5-subsets, a bound `34 ≤`.

**The propagation.** The Schönheim recurrence states that a covering of `v` points by blocks of `k` requires at least `⌈v/k × C(v−1,k−1,t−1)⌉` blocks. Applied to C(25,15,5) with the bound 20, it gives `⌈25 × 20 / 15⌉ = 34`. Applied once more, it gives C(26,16,6) → 56, C(27,17,7) → 89, C(28,18,8) → 139 — the values the upstream repository announces as well. Every `concordance : True` confronts our computation with the upstream sentence.

Those last three numbers are **not** proven by the package: they are the arithmetic consequence of the Schönheim bound, applied by us. The package does not formalise them — the upstream says so explicitly.

## 6. Reading: what « C(24,14,4) ≥ 20 » says, and what it does not say

The two theorems answer each other, and it is worth separating them:

| Theorem | Statement | What it establishes |
|---|---|---|
| `covering_24_14_4` | `20 ≤ 𝒯.card` | every family of blocks of 14 covering the 4-subsets of 24 points has **at least** 20 blocks |
| `covering_25_15_5` | `34 ≤ 𝒯.card` | every family of blocks of 15 covering the 5-subsets of 25 points has **at least** 34 blocks |

The second is a corollary of the first: it goes through the one-point derivatives and the Schönheim bound. It costs no second argument.

What it does **not** say:

- **neither** that `C(24,14,4) = 20`. A lower bound of 20 leaves the values 20, 21, 22 and 23 open; the best known construction needs 23. The gap between the two is precisely the object of the open question.
- **neither** that the covering number is determined. The upstream repository is explicit: it establishes bounds, it determines neither covering number.
- **neither** that the bound 34 for C(25,15,5) is optimal. The best known construction needs 42; the second bound is 32.

The interest of the formalisation is not the number — one more point on a bound known for decades of combinatorics. It lies in the **traceability**: the chain of deductions (degree and incidence counts, rank arguments, rigidity results, then the exclusion of the remaining configurations) is written in Lean and checked by the kernel. A mistake in a base case along one of those steps would not pass unnoticed, whereas it would most likely pass unnoticed in a paper.

This is the formal side that was missing from the notebook [Complexity-08d](../../InformationCalcul/Complexity/Complexity-08d-IndependanceProuvee-BusyBeaver-Covering-Python.ipynb): that one presented the bounds and announced their branching; this one branches them.

### Exercise 1 — The audit predicate

In [7]:
# Exercice 1 : ecrire le predicat d'audit qui refuserait un axiome interdit.
def audit_concluant(lignes):
    # TODO etudiant : renvoyer True si tous les axiomes cites sont dans AUTORISES.
    resultat = None  # TODO etudiant
    return resultat


print("Exercice a completer : audit_concluant(lignes_axiomes)")

Exercice a completer : audit_concluant(lignes_axiomes)


### Exercise 2 — The Schönheim recurrence

In [8]:
# Exercice 2 : ecrire la borne de Schonheim C(v,k,t) >= ceil(v/k * borne interieure).
def borne_de_schonheim(v, k, borne_interieure):
    # TODO etudiant : renvoyer le plus petit entier superieur ou egal a
    # (v * borne_interieure) / k.
    resultat = None  # TODO etudiant
    return resultat


print("Exercice a completer : borne_de_schonheim(25, 15, 20)")

Exercice a completer : borne_de_schonheim(25, 15, 20)


### Exercise 3 — Lower bound and covering number

In [9]:
# Exercice 3 : distinguer une borne inferieure prouvee d'un nombre de couverture.
def classe_du_resultat(borne_inferieure, meilleure_couverture_connue):
    # TODO etudiant : renvoyer "borne inferieure" si le resultat ne donne qu'un
    # plancher, "nombre de couverture" s'il determine exactement C(v,k,t).
    resultat = None  # TODO etudiant
    return resultat


print("Exercice a completer : classe_du_resultat(20, 23)")

Exercice a completer : classe_du_resultat(20, 23)


## 7. Conclusion

- The `covering` package is consumed **as is**, at the pinned commit, by the `from git` clause: no fork, no copy, no adaptation.
- The build passes under the bounded organ, and the axiom audit quotes only `propext`, `Classical.choice`, `Quot.sound` — a set **concordant** with the list the upstream publishes itself in `comparator.json`. No `sorryAx`, no `native_decide`.
- The result is a **lower bound**: `C(24,14,4) ≥ 20` and `C(25,15,5) ≥ 34`. The covering numbers themselves remain undetermined, and the upstream repository says so.
- The figures are read in the consumed package, at the pinned revision: the bounds table, the constants of the statements, and the Schönheim propagation recomputed then confronted with the values announced by the upstream.

### References

- Paper *A structural lower bound for C(24,14,4)* (Apache-2.0) : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Automata\Complexity Theory and Hardness\2026 - Chang - A structural lower bound for C(24,14,4) (github jamesyc-covering, Apache-2.0, Lean formalise).pdf`
- Repository : [jamesyc/covering](https://github.com/jamesyc/covering), commit `e28962cff844f155ca5a5cfb22563083369b490e`
- Companion site : <https://jamesyc.com/covering/>
- Palomar registry : `PALOMAR-2026-10-04-000003`
- Series : [Complexity-08d — Proven independence, Busy Beaver and coverings](../../InformationCalcul/Complexity/Complexity-08d-IndependanceProuvee-BusyBeaver-Covering-Python.ipynb)
- Series issue : #27477 (milestone 3 — Greffe B)